In [109]:
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
import math
import re
import collections
from typing import Dict, List, Tuple

In [3]:
sentence = "The quick brown fox jumped over the lazy dog."

### Utility Function

In [ ]:
def split_chars(text):
    return re.findall(r".", text)

In [12]:
def split_words(text):
    return re.findall(r"[\w]+|[.,!?;]", text)

In [13]:
split_chars(sentence)

['T',
 'h',
 'e',
 ' ',
 'q',
 'u',
 'i',
 'c',
 'k',
 ' ',
 'b',
 'r',
 'o',
 'w',
 'n',
 ' ',
 'f',
 'o',
 'x',
 ' ',
 'j',
 'u',
 'm',
 'p',
 'e',
 'd',
 ' ',
 'o',
 'v',
 'e',
 'r',
 ' ',
 't',
 'h',
 'e',
 ' ',
 'l',
 'a',
 'z',
 'y',
 ' ',
 'd',
 'o',
 'g',
 '.']

In [14]:
split_words(sentence)

['The', 'quick', 'brown', 'fox', 'jumped', 'over', 'the', 'lazy', 'dog', '.']

### Using Ascii

In [ ]:
"""
1. Split each char into list
2. convert each char to ASCII
"""

In [7]:
chars = split_chars(sentence)
len(chars)

45

In [10]:
for char in chars:
    print(char, end=",")

T,h,e, ,q,u,i,c,k, ,b,r,o,w,n, ,f,o,x, ,j,u,m,p,e,d, ,o,v,e,r, ,t,h,e, ,l,a,z,y, ,d,o,g,.,

In [11]:
code = list(map(ord, chars))
code

[84,
 104,
 101,
 32,
 113,
 117,
 105,
 99,
 107,
 32,
 98,
 114,
 111,
 119,
 110,
 32,
 102,
 111,
 120,
 32,
 106,
 117,
 109,
 112,
 101,
 100,
 32,
 111,
 118,
 101,
 114,
 32,
 116,
 104,
 101,
 32,
 108,
 97,
 122,
 121,
 32,
 100,
 111,
 103,
 46]

### Using vocabulary

In [25]:
words = split_words("The quick brown fox jumped over the lazy dog.")
vocabulary = {"[UNK]": 0, " ": 0}
token_num = 1
for token in words:
    if token.lower() not in vocabulary:
        vocabulary[token.lower()] = token_num
        token_num += 1
vocabulary

{'[UNK]': 0,
 ' ': 0,
 'the': 1,
 'quick': 2,
 'brown': 3,
 'fox': 4,
 'jumped': 5,
 'over': 6,
 'lazy': 7,
 'dog': 8,
 '.': 9}

In [28]:
# predefined
vocabulary = {
"[UNK]": 0,
"the": 1,
"quick": 2,
"brown": 3,
"fox": 4,
"jumped": 5,
"over": 6,
"dog": 7,
".": 8,
}

In [29]:
words = split_words("The quick brown fox jumped over the lazy dog.")
words

['The', 'quick', 'brown', 'fox', 'jumped', 'over', 'the', 'lazy', 'dog', '.']

In [30]:
indices = [vocabulary.get(word, vocabulary.get("[UNK]")) for word in words]
indices

[0, 2, 3, 4, 5, 6, 1, 0, 7, 8]

### Character-level Tokenizer

In [45]:
import collections
def compute_char_vocabulary(inputs, max_size):
    char_counts = collections.Counter()
    for x in inputs:
        x = x.lower()
        tokens = re.findall(r".", x)
        char_counts.update(tokens)
    vocabulary = ["[UNK]"]
    most_common = char_counts.most_common(max_size - len(vocabulary))
    for token, count in most_common:
        vocabulary.append(token)
    return dict((token, i) for i, token in enumerate(vocabulary))

compute_char_vocabulary(sentence, 20)

{'[UNK]': 0,
 ' ': 1,
 'e': 2,
 'o': 3,
 't': 4,
 'h': 5,
 'u': 6,
 'r': 7,
 'd': 8,
 'q': 9,
 'i': 10,
 'c': 11,
 'k': 12,
 'b': 13,
 'w': 14,
 'n': 15,
 'f': 16,
 'x': 17,
 'j': 18,
 'm': 19}

In [40]:
class CharTokenizer:
    def __init__(self, vocabulary = {"[UNK]": 0}):
        self.vocabulary = vocabulary
        self.unk_id = vocabulary["[UNK]"]

    def build_vocabulary(self, inputs):
        tokens = self.split(inputs)
        token_num = 1
        for token in tokens:
            if token.lower() not in self.vocabulary:
                self.vocabulary[token.lower()] = token_num
                token_num += 1

    def standardize(self, inputs):
        return inputs.lower()

    def split(self, inputs):
        return re.findall(r".", inputs)

    def index(self, tokens):
        return [self.vocabulary.get(t, self.unk_id) for t in tokens]

    def __call__(self, inputs):
        self.build_vocabulary(inputs)
        inputs = self.standardize(inputs)
        tokens = self.split(inputs)
        return self.index(tokens)

In [41]:
char_tokenizer = CharTokenizer()
char_tokenizer(sentence)

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 4,
 10,
 11,
 12,
 13,
 14,
 4,
 15,
 12,
 16,
 4,
 17,
 6,
 18,
 19,
 3,
 20,
 4,
 12,
 21,
 3,
 11,
 4,
 1,
 2,
 3,
 4,
 22,
 23,
 24,
 25,
 4,
 20,
 12,
 26,
 27]

### World-level Tokenizer

In [57]:
def compute_word_vocabulary(inputs, max_size):
    word_counts = collections.Counter()
    for x in inputs:
        x = x.lower()
        tokens = re.findall(r"[\w]+|[.,!?;]", x)
        word_counts.update(tokens)
    vocabulary = ["[UNK]"]
    most_common = word_counts.most_common(max_size - len(vocabulary))
    for token, count in most_common:
        vocabulary.append(token)
    return dict((token, i) for i, token in enumerate(vocabulary))

compute_word_vocabulary([sentence], 10)

{'[UNK]': 0,
 'the': 1,
 'quick': 2,
 'brown': 3,
 'fox': 4,
 'jumped': 5,
 'over': 6,
 'lazy': 7,
 'dog': 8,
 '.': 9}

In [50]:
class WordTokenizer:
    def __init__(self, vocabulary = {"[UNK]": 0}):
        self.vocabulary = vocabulary
        self.unk_id = vocabulary["[UNK]"]

    def build_vocabulary(self, inputs):
        tokens = self.split(inputs)
        token_num = 1
        for token in tokens:
            if token.lower() not in self.vocabulary:
                self.vocabulary[token.lower()] = token_num
                token_num += 1

    def standardize(self, inputs):
        return inputs.lower()

    def split(self, inputs):
        return re.findall(r"[\w]+|[.,!?;]", inputs)

    def index(self, tokens):
        return [self.vocabulary.get(t, self.unk_id) for t in tokens]

    def __call__(self, inputs):
        self.build_vocabulary(inputs)
        inputs = self.standardize(inputs)
        tokens = self.split(inputs)
        indices = self.index(tokens)
        return indices

In [51]:
word_tokenizer = WordTokenizer()
word_tokenizer(sentence)

[1, 2, 3, 4, 5, 6, 1, 7, 8, 9]

### Use Case

#### Load dataset

In [59]:
# Moby Dick Full book as text file
filename = keras.utils.get_file(origin="https://www.gutenberg.org/cache/epub/2701/pg2701.txt")
filename

1276263/1276263 ━━━━━━━━━━━━━━━━━━━━ 2s 1us/step


'C:\\Users\\PRASHANTH N\\.keras\\datasets\\pg2701.txt'

In [61]:
moby_dick = list(open(filename, "r", encoding="utf-8"))
moby_dick

['The Project Gutenberg eBook of Moby Dick; Or, The Whale\n',
 '    \n',
 'This eBook is for the use of anyone anywhere in the United States and\n',
 'most other parts of the world at no cost and with almost no restrictions\n',
 'whatsoever. You may copy it, give it away or re-use it under the terms\n',
 'of the Project Gutenberg License included with this eBook or online\n',
 'at www.gutenberg.org. If you are not located in the United States,\n',
 'you will have to check the laws of the country where you are located\n',
 'before using this eBook.\n',
 '\n',
 'Title: Moby Dick; Or, The Whale\n',
 '\n',
 'Author: Herman Melville\n',
 '\n',
 '\n',
 '        \n',
 'Release date: July 1, 2001 [eBook #2701]\n',
 '                Most recently updated: February 10, 2026\n',
 '\n',
 'Language: English\n',
 '\n',
 'Other information and formats: www.gutenberg.org/ebooks/2701\n',
 '\n',
 'Credits: Daniel Lazarus, Jonesey, and David Widger\n',
 '\n',
 '\n',
 '*** START OF THE PROJECT GUTENBERG E

#### Char Level Tokenizer

In [69]:
vocabulary = compute_char_vocabulary(moby_dick, max_size=1000)
vocabulary

{'[UNK]': 0,
 ' ': 1,
 'e': 2,
 't': 3,
 'a': 4,
 'o': 5,
 'n': 6,
 'i': 7,
 's': 8,
 'h': 9,
 'r': 10,
 'l': 11,
 'd': 12,
 'u': 13,
 'm': 14,
 'c': 15,
 'w': 16,
 'g': 17,
 'f': 18,
 ',': 19,
 'p': 20,
 'y': 21,
 'b': 22,
 'v': 23,
 'k': 24,
 '.': 25,
 ';': 26,
 '’': 27,
 '-': 28,
 '!': 29,
 '—': 30,
 '“': 31,
 'q': 32,
 '”': 33,
 'j': 34,
 'x': 35,
 '?': 36,
 '_': 37,
 'z': 38,
 '1': 39,
 '(': 40,
 ')': 41,
 ':': 42,
 '0': 43,
 '‘': 44,
 '2': 45,
 '3': 46,
 '8': 47,
 '*': 48,
 '5': 49,
 '7': 50,
 '4': 51,
 '6': 52,
 '9': 53,
 'æ': 54,
 '™': 55,
 '/': 56,
 'œ': 57,
 'é': 58,
 '£': 59,
 '$': 60,
 '•': 61,
 '[': 62,
 ']': 63,
 'è': 64,
 '#': 65,
 '&': 66,
 '\u200f': 67,
 'ח': 68,
 'ו': 69,
 '\u200e': 70,
 'ϰ': 71,
 'η': 72,
 'τ': 73,
 'ο': 74,
 'ς': 75,
 'â': 76,
 '%': 77,
 '+': 78}

In [70]:
char_tokenizer = CharTokenizer(vocabulary)
print("Vocabulary length:", len(vocabulary))
print("Vocabulary start:", list(vocabulary.keys())[:10])
print("Vocabulary end:", list(vocabulary.keys())[-10:])
print("Line length:", len(char_tokenizer("Call me Ishmael. Some years ago--never mind how long precisely.")))

Vocabulary length: 79
Vocabulary start: ['[UNK]', ' ', 'e', 't', 'a', 'o', 'n', 'i', 's', 'h']
Vocabulary end: ['ו', '\u200e', 'ϰ', 'η', 'τ', 'ο', 'ς', 'â', '%', '+']
Line length: 63


#### Word Level Tokenizer

In [76]:
vocabulary = compute_word_vocabulary(moby_dick, max_size=20000)
word_tokenizer = WordTokenizer(vocabulary)

In [77]:
vocabulary

{'[UNK]': 0,
 ',': 1,
 'the': 2,
 '.': 3,
 'of': 4,
 'and': 5,
 'a': 6,
 'to': 7,
 'in': 8,
 ';': 9,
 'that': 10,
 'it': 11,
 'his': 12,
 'i': 13,
 'he': 14,
 'but': 15,
 's': 16,
 'with': 17,
 '!': 18,
 'as': 19,
 'is': 20,
 'was': 21,
 'for': 22,
 'all': 23,
 'this': 24,
 'at': 25,
 'whale': 26,
 'by': 27,
 'not': 28,
 'from': 29,
 'on': 30,
 'so': 31,
 'him': 32,
 'be': 33,
 '?': 34,
 'you': 35,
 'one': 36,
 'there': 37,
 'or': 38,
 'now': 39,
 'had': 40,
 'have': 41,
 'were': 42,
 'they': 43,
 'which': 44,
 'like': 45,
 'then': 46,
 'me': 47,
 'are': 48,
 'their': 49,
 'some': 50,
 'what': 51,
 'when': 52,
 'an': 53,
 'no': 54,
 'my': 55,
 'upon': 56,
 'out': 57,
 'man': 58,
 'up': 59,
 'into': 60,
 'ship': 61,
 'ahab': 62,
 'more': 63,
 'if': 64,
 'them': 65,
 'ye': 66,
 'we': 67,
 'sea': 68,
 'old': 69,
 'other': 70,
 'would': 71,
 'been': 72,
 'over': 73,
 'these': 74,
 'will': 75,
 'though': 76,
 'down': 77,
 'its': 78,
 'only': 79,
 'such': 80,
 'who': 81,
 'any': 82,
 'head':

In [80]:
print("Vocabulary length:", len(vocabulary))
print("Vocabulary start:", list(vocabulary.keys())[:15])
print("Vocabulary end:", list(vocabulary.keys())[-10:])
print("Line length:", len(word_tokenizer("Call me Ishmael. Some years ago--never mind how long precisely.")))

Vocabulary length: 17644
Vocabulary start: ['[UNK]', ',', 'the', '.', 'of', 'and', 'a', 'to', 'in', ';', 'that', 'it', 'his', 'i', 'he']
Vocabulary end: ['checks', 'hart', 'originator', 'network', 'volunteer', 'confirmed', 'pg', 'includes', 'subscribe', 'newsletter']
Line length: 13


### Subword Tokenization - Byte pair Encoding

In [162]:
data = [
    "the quick brown fox",
    "the slow brown fox",
    "the quick brown foxhound",
]

In [163]:
# Initializing state for the byte-pair encoding algorithm
def count_and_split_words(data):
    counts = collections.Counter()
    for line in data:
        line = line.lower()
        for word in re.findall(r"[\w]+|[.,!?;]", line):
            chars = re.findall(r".", word)
            split_word = " ".join(chars)
            counts[split_word] += 1
    return dict(counts)
counts = count_and_split_words(data)
counts

{'t h e': 3,
 'q u i c k': 2,
 'b r o w n': 3,
 'f o x': 2,
 's l o w': 1,
 'f o x h o u n d': 1}

In [164]:
# Running a few steps of byte-pair merging

def count_pairs(counts):
    pairs = collections.Counter()
    for word, freq in counts.items():
        symbols = word.split()
        for pair in zip(symbols[:-1], symbols[1:]):
            pairs[pair] += freq
    return pairs

def merge_pair(counts, first, second):
    split = re.compile(f"(?<!\\S){first} {second}(?!\\S)")
    merged = f"{first}{second}"
    return {split.sub(merged, word): count for word, count in counts.items()}

In [165]:
pairs = None
while pairs is None or len(pairs)>1:
    pairs = count_pairs(counts)
    first, second = max(pairs, key=pairs.get)
    counts = merge_pair(counts, first, second)
    print(list(counts.keys()))

['t h e', 'q u i c k', 'b r ow n', 'f o x', 's l ow', 'f o x h o u n d']
['th e', 'q u i c k', 'b r ow n', 'f o x', 's l ow', 'f o x h o u n d']
['the', 'q u i c k', 'b r ow n', 'f o x', 's l ow', 'f o x h o u n d']
['the', 'q u i c k', 'br ow n', 'f o x', 's l ow', 'f o x h o u n d']
['the', 'q u i c k', 'brow n', 'f o x', 's l ow', 'f o x h o u n d']
['the', 'q u i c k', 'brown', 'f o x', 's l ow', 'f o x h o u n d']
['the', 'q u i c k', 'brown', 'fo x', 's l ow', 'fo x h o u n d']
['the', 'q u i c k', 'brown', 'fox', 's l ow', 'fox h o u n d']
['the', 'qu i c k', 'brown', 'fox', 's l ow', 'fox h o u n d']
['the', 'qui c k', 'brown', 'fox', 's l ow', 'fox h o u n d']
['the', 'quic k', 'brown', 'fox', 's l ow', 'fox h o u n d']
['the', 'quick', 'brown', 'fox', 's l ow', 'fox h o u n d']
['the', 'quick', 'brown', 'fox', 'sl ow', 'fox h o u n d']
['the', 'quick', 'brown', 'fox', 'slow', 'fox h o u n d']
['the', 'quick', 'brown', 'fox', 'slow', 'foxh o u n d']
['the', 'quick', 'brown', '